# 文本相似度实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

## Step2 加载数据集

In [2]:
dataset = load_dataset("json", data_files="./train_pair_1w.json", split="train")
dataset

Dataset({
    features: ['sentence1', 'sentence2', 'label'],
    num_rows: 10000
})

In [3]:
dataset[0]

{'sentence1': '找一部小时候的动画片', 'sentence2': '求一部小时候的动画片。谢了', 'label': '1'}

## Step3 划分数据集

In [4]:
datasets = dataset.train_test_split(test_size=0.2)
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 2000
    })
})

## Step4 数据集预处理

In [5]:
import torch

tokenizer = AutoTokenizer.from_pretrained("../../model/hfl/chinese-macbert-base")

def process_function(examples):
    sentences = []
    labels = []
    for sen1, sen2, label in zip(examples["sentence1"], examples["sentence2"], examples["label"]):
        sentences.append(sen1)
        sentences.append(sen2)
        labels.append(1 if int(label) == 1 else -1)
    # input_ids, attention_mask, token_type_ids
    tokenized_examples = tokenizer(sentences, max_length=128, truncation=True, padding="max_length")
    tokenized_examples = {k: [v[i: i + 2] for i in range(0, len(v), 2)] for k, v in tokenized_examples.items()}
    tokenized_examples["labels"] = labels
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [6]:
print(tokenized_datasets["train"][0])

{'input_ids': [[101, 6821, 763, 782, 4802, 2141, 808, 782, 7410, 1838, 8024, 4493, 5635, 6375, 782, 2630, 4125, 8024, 852, 3172, 881, 714, 2692, 6399, 1168, 8024, 5735, 679, 3221, 4382, 1065, 1984, 6152, 2844, 1961, 2400, 6813, 6862, 7023, 1357, 749, 677, 6835, 6121, 1220, 8024, 2607, 2586, 1059, 1814, 4638, 782, 6963, 833, 2190, 1961, 2584, 4680, 5445, 6228, 8024, 1961, 738, 3193, 2218, 6158, 782, 812, 2792, 6890, 2461, 749, 511, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 4448, 2548, 1358, 1168, 1059, 1814, 782, 4638, 3918, 1147, 1068, 2552, 4638, 1398, 2658, 8024, 5445, 800, 2190, 3634, 3188, 679, 3209, 4635, 738, 679, 1762, 725, 749, 8024, 2590, 1649, 711, 1059, 1814, 782, 2792, 1328, 2626, 8024, 852, 1961, 1316, 4495, 2398, 5018, 671, 3613, 2697, 1168, 7444, 6206, 5439, 3301, 1351, 812, 4638, 1068, 1147, 749, 511, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

## Step5 创建模型

In [7]:
from transformers import BertForSequenceClassification, BertPreTrainedModel, BertModel
from typing import Optional
from transformers.configuration_utils import PretrainedConfig
from torch.nn import CosineSimilarity, CosineEmbeddingLoss

class DualModel(BertPreTrainedModel):

    def __init__(self, config: PretrainedConfig, *inputs, **kwargs):
        super().__init__(config, *inputs, **kwargs)
        self.bert = BertModel(config)
        self.post_init()

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        # Step1 分别获取sentenceA 和 sentenceB的输入
        senA_input_ids, senB_input_ids = input_ids[:, 0], input_ids[:, 1]
        senA_attention_mask, senB_attention_mask = attention_mask[:, 0], attention_mask[:, 1]
        senA_token_type_ids, senB_token_type_ids = token_type_ids[:, 0], token_type_ids[:, 1]

        # Step2 分别获取sentenceA 和 sentenceB的向量表示
        senA_outputs = self.bert(
            senA_input_ids,
            attention_mask=senA_attention_mask,
            token_type_ids=senA_token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        senA_pooled_output = senA_outputs[1]    # [batch, hidden]

        senB_outputs = self.bert(
            senB_input_ids,
            attention_mask=senB_attention_mask,
            token_type_ids=senB_token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        senB_pooled_output = senB_outputs[1]    # [batch, hidden]

        # step3 计算相似度

        cos = CosineSimilarity()(senA_pooled_output, senB_pooled_output)    # [batch, ]

        # step4 计算loss

        loss = None
        if labels is not None:
            loss_fct = CosineEmbeddingLoss(0.3)
            loss = loss_fct(senA_pooled_output, senB_pooled_output, labels)

        output = (cos,)
        return ((loss,) + output) if loss is not None else output
    
model = DualModel.from_pretrained("../../model/hfl/chinese-macbert-base")

## Step6 创建评估函数

In [8]:
import evaluate

acc_metric = evaluate.load("./metric_accuracy.py")
f1_metirc = evaluate.load("../../evaluate/metrics/f1")

In [9]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    
    predictions = (predictions > 0.7).astype(int) 
    labels = (labels > 0).astype(int)
    
    # predictions = [int(p > 0.7) for p in predictions]
    # labels = [int(l > 0) for l in labels]
    
    # predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metirc.compute(predictions=predictions, references=labels)
    acc.update(f1)
    return acc

## Step7 创建TrainingArguments

In [10]:
train_args = TrainingArguments(output_dir="./dual_model",      # 输出文件夹
                               per_device_train_batch_size=16,  # 训练时的batch_size
                               gradient_accumulation_steps=2,   # 梯度累积步数
                               per_device_eval_batch_size=16,   # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               eval_strategy="epoch",           # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True,
                               num_train_epochs=1) 


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step8 创建Trainer

In [11]:
trainer = Trainer(model=model, 
                  args=train_args, 
                  tokenizer=tokenizer,
                  train_dataset=tokenized_datasets["train"], 
                  eval_dataset=tokenized_datasets["test"], 
                  compute_metrics=eval_metric)

## Step9 模型训练

In [12]:
trainer.train()

  0%|          | 0/250 [00:00<?, ?it/s]

{'loss': 0.3553, 'grad_norm': 3.987112045288086, 'learning_rate': 1.9200000000000003e-05, 'epoch': 0.04}
{'loss': 0.293, 'grad_norm': 4.8557562828063965, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.08}
{'loss': 0.2589, 'grad_norm': 4.207776069641113, 'learning_rate': 1.76e-05, 'epoch': 0.12}
{'loss': 0.3071, 'grad_norm': 3.6964218616485596, 'learning_rate': 1.6800000000000002e-05, 'epoch': 0.16}
{'loss': 0.2907, 'grad_norm': 3.5061941146850586, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.2}
{'loss': 0.2697, 'grad_norm': 3.9466447830200195, 'learning_rate': 1.5200000000000002e-05, 'epoch': 0.24}
{'loss': 0.2426, 'grad_norm': 4.2203168869018555, 'learning_rate': 1.4400000000000001e-05, 'epoch': 0.28}
{'loss': 0.2646, 'grad_norm': 4.003051280975342, 'learning_rate': 1.3600000000000002e-05, 'epoch': 0.32}
{'loss': 0.2579, 'grad_norm': 3.945965051651001, 'learning_rate': 1.2800000000000001e-05, 'epoch': 0.36}
{'loss': 0.258, 'grad_norm': 4.468891620635986, 'learning_rate': 1

  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.21199335157871246, 'eval_accuracy': 0.739, 'eval_f1': 0.7023945267958951, 'eval_runtime': 24.942, 'eval_samples_per_second': 80.186, 'eval_steps_per_second': 5.012, 'epoch': 1.0}
{'train_runtime': 328.6539, 'train_samples_per_second': 24.342, 'train_steps_per_second': 0.761, 'train_loss': 0.253209716796875, 'epoch': 1.0}


TrainOutput(global_step=250, training_loss=0.253209716796875, metrics={'train_runtime': 328.6539, 'train_samples_per_second': 24.342, 'train_steps_per_second': 0.761, 'total_flos': 1052425322496000.0, 'train_loss': 0.253209716796875, 'epoch': 1.0})

## Step10 模型评估

In [13]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.21199335157871246,
 'eval_accuracy': 0.739,
 'eval_f1': 0.7023945267958951,
 'eval_runtime': 24.7081,
 'eval_samples_per_second': 80.945,
 'eval_steps_per_second': 5.059,
 'epoch': 1.0}

## Step11 模型预测

In [14]:
class SentenceSimilarityPipeline:

    def __init__(self, model, tokenizer) -> None:
        self.model = model.bert
        self.tokenizer = tokenizer
        self.device = model.device

    def preprocess(self, senA, senB):
        return self.tokenizer([senA, senB], max_length=128, truncation=True, return_tensors="pt", padding=True)

    def predict(self, inputs):
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        return self.model(**inputs)[1]  # [2, 768]

    def postprocess(self, logits):
        cos = CosineSimilarity()(logits[None, 0, :], logits[None,1, :]).squeeze().cpu().item()
        return cos

    def __call__(self, senA, senB, return_vector=False):
        inputs = self.preprocess(senA, senB)
        logits = self.predict(inputs)
        result = self.postprocess(logits)
        if return_vector:
            return result, logits
        else:
            return result

In [15]:
pipe = SentenceSimilarityPipeline(model, tokenizer)

In [28]:
pipe("我喜欢北京", "阿山东龙口港", return_vector=True)

(0.2765252888202667,
 tensor([[-0.9960, -0.9981,  0.4191,  ...,  0.9887,  0.7374, -0.8895],
         [ 0.9807, -0.3784,  1.0000,  ..., -0.9980, -0.9841, -0.8376]],
        device='cuda:0', grad_fn=<TanhBackward0>))